<a href="https://colab.research.google.com/github/K-Sanyam-130/AI_LAB/blob/main/lab7_FOL.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import re

def is_variable(x):
    # A variable is a string that starts with a lowercase letter
    return isinstance(x, str) and len(x) > 0 and x[0].islower() and '(' not in x

def is_constant(x):
    # A constant is a string that starts with an uppercase letter
    return isinstance(x, str) and len(x) > 0 and x[0].isupper() and '(' not in x

def is_compound(x):
    # A compound/functional term is represented as a tuple: (functor, arg1, arg2, ...)
    return isinstance(x, tuple) and len(x) > 0

def occurs_check(var, term, substitution):
    if var == term:
        return True
    if is_variable(term) and term in substitution:
        return occurs_check(var, substitution[term], substitution)
    if is_compound(term):
        return any(occurs_check(var, arg, substitution) for arg in term[1:])
    return False

def unify(x, y, substitution=None):
    if substitution is None:
        substitution = {}

    # Resolve variables to their current bindings
    while is_variable(x) and x in substitution:
        x = substitution[x]
    while is_variable(y) and y in substitution:
        y = substitution[y]

    if x == y:
        return substitution

    if is_variable(x):
        if occurs_check(x, y, substitution):
            return None
        substitution[x] = y
        return substitution

    if is_variable(y):
        if occurs_check(y, x, substitution):
            return None
        substitution[y] = x
        return substitution

    if is_compound(x) and is_compound(y):
        if x[0] != y[0] or len(x) != len(y):
            return None
        for a, b in zip(x[1:], y[1:]):
            substitution = unify(a, b, substitution)
            if substitution is None:
                return None
        return substitution

    return None

def tokenize(expr_str):
    # Helper to split expression into tokens including words, parentheses, and commas
    return re.findall(r'[a-zA-Z0-9_]+|\(|\)|,', expr_str)

class Parser:
    def __init__(self, tokens):
        self.tokens = tokens
        self.pos = 0

    def peek(self):
        return self.tokens[self.pos] if self.pos < len(self.tokens) else None

    def next_token(self):
        if self.pos >= len(self.tokens):
            raise SyntaxError("Unexpected end of input (check your parentheses!)")
        token = self.peek()
        self.pos += 1
        return token

    def parse(self):
        token = self.next_token()
        if self.peek() == '(':
            self.next_token() # Consume '('
            args = []
            while self.peek() != ')':
                if self.peek() is None:
                    raise SyntaxError("Mismatched parentheses: expected ')'")
                args.append(self.parse())
                if self.peek() == ',':
                    self.next_token() # Consume ','
            self.next_token() # Consume ')'
            return tuple([token] + args)
        return token

def parse_expression(expr_str):
    # Clean outer quotation marks if any, and parse into nested representation
    clean_str = expr_str.strip().strip("'\"")
    tokens = tokenize(clean_str)
    if not tokens:
        raise ValueError("Empty expression")
    return Parser(tokens).parse()

print("--- Robust First-Order Logic Unification Algorithm ---")
print("Please enter expressions in prefix logic format, e.g., p(B,x,f(g(z)))")
print("Note: lowercase terms like 'x', 'y', 'z' are variables. Uppercase terms like 'B', 'John' are constants.\n")

try:
    user_x = input("Enter Expression 1: ")
    user_y = input("Enter Expression 2: ")

    expr_x = parse_expression(user_x)
    expr_y = parse_expression(user_y)

    print(f"\nParsed Expression 1: {expr_x}")
    print(f"Parsed Expression 2: {expr_y}")

    result = unify(expr_x, expr_y)

    if result is not None:
        print("\nUnification Successful!")
        # Standardize the output format for representation
        readable_substitution = {k: str(v).replace("'", "").replace(" ", "") for k, v in result.items()}
        print("Substitution:", readable_substitution)
    else:
        print("\nUnification Failed! (Occur Check, Constant Mismatch, or Shape Mismatch)")
except Exception as e:
    print(f"\nAn error occurred while parsing input: {e}")

--- Robust First-Order Logic Unification Algorithm ---
Please enter expressions in prefix logic format, e.g., p(B,x,f(g(z)))
Note: lowercase terms like 'x', 'y', 'z' are variables. Uppercase terms like 'B', 'John' are constants.

Enter Expression 1: p(B,X,f(g(z)))
Enter Expression 2: p(Z,f(y),f(y))

Parsed Expression 1: ('p', 'B', 'X', ('f', ('g', 'z')))
Parsed Expression 2: ('p', 'Z', ('f', 'y'), ('f', 'y'))

Unification Failed! (Occur Check, Constant Mismatch, or Shape Mismatch)


In [ ]:
import re

class LogicVar:
    def __init__(self, label):
        self.label = label

    def __str__(self):
        return self.label

    def __repr__(self):
        return f"Var({self.label})"

    def __eq__(self, other):
        return isinstance(other, LogicVar) and self.label == other.label

    def __hash__(self):
        return hash(self.label)


class LogicConst:
    def __init__(self, label):
        self.label = label

    def __str__(self):
        return self.label

    def __repr__(self):
        return f"Const({self.label})"

    def __eq__(self, other):
        return isinstance(other, LogicConst) and self.label == other.label

    def __hash__(self):
        return hash(self.label)


class LogicFn:
    def __init__(self, name, parameters):
        self.name = name
        self.parameters = list(parameters)

    def __str__(self):
        return f"{self.name}({', '.join(str(p) for p in self.parameters)})"

    def __repr__(self):
        return f"Fn({self.name}, {self.parameters})"

    def __eq__(self, other):
        return isinstance(other, LogicFn) and self.name == other.name and self.parameters == other.parameters


def check_cycle(variable, term, env):
    """Occurs check: detects if 'variable' is nested inside 'term' under current environment bindings."""
    if variable == term:
        return True
    if isinstance(term, LogicVar) and term in env:
        return check_cycle(variable, env[term], env)
    if isinstance(term, LogicFn):
        return any(check_cycle(variable, child, env) for child in term.parameters)
    return False


def bind_var(variable, expression, env):
    """Binds a LogicVar to an expression if valid, resolving existing bindings and performing occurs checks."""
    if variable in env:
        return solve_unify(env[variable], expression, env)
    if isinstance(expression, LogicVar) and expression in env:
        return solve_unify(variable, env[expression], env)
    if check_cycle(variable, expression, env):
        return None

    updated_env = env.copy()
    updated_env[variable] = expression
    return updated_env


def solve_unify(t1, t2, env=None):
    """Primary logic engine to unify structure t1 and t2 with substitution dictionary 'env'."""
    if env is None:
        env = {}

    if t1 == t2:
        return env
    if isinstance(t1, LogicVar):
        return bind_var(t1, t2, env)
    if isinstance(t2, LogicVar):
        return bind_var(t2, t1, env)

    if isinstance(t1, LogicFn) and isinstance(t2, LogicFn):
        if t1.name != t2.name or len(t1.parameters) != len(t2.parameters):
            return None
        current_env = env
        for p1, p2 in zip(t1.parameters, t2.parameters):
            current_env = solve_unify(p1, p2, current_env)
            if current_env is None:
                return None
        return current_env

    return None


def tokenize_and_parse(token_str):
    """Converts a logical string representation into nested instances of LogicVar, LogicConst, or LogicFn."""
    token_str = token_str.strip().replace(" ", "")
    if not token_str:
        return None

    outer_pattern = re.match(r'^([a-zA-Z0-9_]+)\((.*)\)$', token_str)
    if outer_pattern:
        fn_name = outer_pattern.group(1)
        nested_str = outer_pattern.group(2)

        elements = []
        buffer = []
        depth = 0
        for char in nested_str:
            if char == '(':
                depth += 1
            elif char == ')':
                depth -= 1

            if char == ',' and depth == 0:
                elements.append(tokenize_and_parse("".join(buffer)))
                buffer = []
            else:
                buffer.append(char)
        if buffer:
            elements.append(tokenize_and_parse("".join(buffer)))

        return LogicFn(fn_name, elements)

    if token_str[0].isupper():
        return LogicVar(token_str)
    return LogicConst(token_str)


def start_terminal_app():
    """Interactive console interface taking raw inputs from user and matching them."""
    print("=================== LOGICAL UNIFICATION TERMINAL ===================")
    print("Note:")
    print(" - Variables must start with UPPERCASE characters (e.g. X, Y, Z)")
    print(" - Constants & Functions must start with lowercase characters (e.g. a, f(X))")
    print(" - Type 'exit' to terminate the app.\n")

    while True:
        stmt1 = input("Enter First Statement: ").strip()
        if stmt1.lower() == 'exit' or not stmt1:
            print("Terminating. Goodbye!")
            break

        stmt2 = input("Enter Second Statement: ").strip()
        if stmt2.lower() == 'exit' or not stmt2:
            print("Terminating. Goodbye!")
            break

        try:
            ast1 = tokenize_and_parse(stmt1)
            ast2 = tokenize_and_parse(stmt2)

            print(f"Parsed First: {ast1}")
            print(f"Parsed Second: {ast2}")

            results = solve_unify(ast1, ast2)
            if results is not None:
                clean_bindings = {str(k): str(v) for k, v in results.items()}
                print("RESULT: SUCCESSFUL UNIFICATION!")
                print(f"SUBSTITUTIONS: {clean_bindings}\n")
            else:
                print("RESULT: FAILED TO UNIFY (No compatible substitution exists).\n")
        except Exception as error:
            print(f"Error processing terms: {error}. Please check syntax formatting.\n")

# Launch the program inside Colab terminal input context
start_terminal_app()



=================== LOGICAL UNIFICATION TERMINAL ===================
Note:
 - Variables must start with UPPERCASE characters (e.g. X, Y, Z)
 - Constants & Functions must start with lowercase characters (e.g. a, f(X))
 - Type 'exit' to terminate the app.

Enter First Statement: p(b,X,f(g(Z)))
Enter Second Statement: p(Z,f(Y),f(Y))
Parsed First: p(b, X, f(g(Z)))
Parsed Second: p(Z, f(Y), f(Y))
RESULT: SUCCESSFUL UNIFICATION!
SUBSTITUTIONS: {'Z': 'b', 'X': 'f(Y)', 'Y': 'g(Z)'}

Enter First Statement: p(f(a),g(Y))
Enter Second Statement: p(X,X)
Parsed First: p(f(a), g(Y))
Parsed Second: p(X, X)
RESULT: FAILED TO UNIFY (No compatible substitution exists).

Enter First Statement: prime(11)
Enter Second Statement: prime(y)
Parsed First: prime(11)
Parsed Second: prime(y)
RESULT: FAILED TO UNIFY (No compatible substitution exists).

Enter First Statement: prime(11)
Enter Second Statement: prime(Y)
Parsed First: prime(11)
Parsed Second: prime(Y)
RESULT: SUCCESSFUL UNIFICATION!
SUBSTITUTIONS: {'Y